In [0]:
%run ../../00_common/data_utils

In [0]:
def load_export_log():
    """
    从 golden_cdp_anonymization_database 读取 t_consumer_anonymization_log，
    过滤 AnonymizationGraceDate <= current_date 的数据。
    """
    anonymization_db = get_env_config('golden_cdp_anonymization_database')
    export_log_table = f"{anonymization_db}.t_consumer_anonymization_log"

    export_log_df = spark.table(export_log_table).where(
        F.to_date(F.col("AnonymizationGraceDate")) <= F.current_date()
    )

    return export_log_df

In [0]:
def load_recently_completed(days=None):
    """
    从 t_mdm_anonymization_log 读取 status=Complete 的记录，
    返回去重后的 (MarketCode, BrandCode, Ukey)，用于排除已处理的 key。

    days 为空/0 时读取全部；否则只取 create_time 在最近 days 天内的记录。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{anonymization_db}.t_mdm_anonymization_log"

    completed_df = spark.table(log_table).where(F.col("status") == ANON_STATUS_COMPLETE)

    if days not in (None, "", "0", 0):
        completed_df = completed_df.where(
            F.col("create_time") >= F.expr(f"current_timestamp() - INTERVAL {days} DAYS")
        )

    return (
        completed_df
        .select(
            F.col("MarketCode"),
            F.col("BrandCode"),
            F.col("Old_UniversalKey").alias("Ukey")
        )
        .distinct()
    )

In [0]:
def apply_rate_limit(exploded_df, config_json):
    """
    按 market 限流。以 (MarketCode, BrandCode, Ukey) 为一个维度，
    每 market 内按 LastActivityTime 升序（从老到新）选取最多 max_job_proccess_count 个维度。
    未配置的 market 全量取走。

    返回 None 表示无任何记录被选中。
    """
    config_list = json.loads(config_json)

    # 限流维度表：每个 (MarketCode, BrandCode, Ukey) 聚合出最早 LastActivityTime
    dim_df = (
        exploded_df
        .groupBy("MarketCode", "BrandCode", "Ukey")
        .agg(F.min("LastActivityTime").alias("min_last_activity_time"))
        .checkpoint(eager=True)
    )

    market_stats_rows = (
        dim_df
        .groupBy("MarketCode")
        .agg(F.count(F.lit(1)).alias("total_pending"))
        .collect()
    )
    market_stats = {row["MarketCode"]: row["total_pending"] for row in market_stats_rows}

    selected_dfs = []
    configured_markets = set()

    for item in config_list:
        market = item["market_code"]
        configured_markets.add(market)

        rate_limit_enable = item.get("rate_limit_enable", False)
        max_count = item.get("max_job_proccess_count", 0)

        total_pending = market_stats.get(market, 0)

        if total_pending == 0:
            print(f"{market}: no records")
            continue

        if rate_limit_enable and max_count >= 0:
            # 从老到新：LastActivityTime 升序，MarketCode/BrandCode/Ukey 确定性 tie-breaker
            selected_dim = (
                dim_df
                .filter(F.col("MarketCode") == market)
                .orderBy(
                    F.col("min_last_activity_time").asc(),
                    F.col("MarketCode").asc(),
                    F.col("BrandCode").asc(),
                    F.col("Ukey").asc()
                )
                .limit(max_count)
                .select("BrandCode", "Ukey")
            )

            selected = (
                exploded_df
                .filter(F.col("MarketCode") == market)
                .join(selected_dim, ["BrandCode", "Ukey"], "inner")
            )

            actual_dims = min(total_pending, max_count)
            print(f"{market}: rate_limit enabled, max={max_count} ukey dims, "
                  f"selected={actual_dims} dims")
        else:
            selected = exploded_df.filter(F.col("MarketCode") == market)
            print(f"{market}: rate_limit disabled, selected {total_pending} ukeys")

        selected_dfs.append(selected)

    # 未配置 market 全量取走
    # other_count = sum(
    #     cnt for m, cnt in market_stats.items()
    #     if m not in configured_markets
    # )
    # if other_count > 0:
    #     other_markets = exploded_df.filter(~F.col("MarketCode").isin(list(configured_markets)))
    #     print(f"other markets (no config): selected {other_count} ukeys")
    #     selected_dfs.append(other_markets)

    if not selected_dfs:
        return None

    selected_df = selected_dfs[0]
    for df in selected_dfs[1:]:
        selected_df = selected_df.union(df)

    return selected_df.distinct()

In [0]:
def match_with_master_consumer(exploded_df):
    """
    对同一个 market + brand + ukey,
    比较 export_log 与 master_consumer 的(SourceSystemCode, ConsumerId) 组合集合是否完全一致，标记 is_matched。
    """

    acs_sources_df = spark.table(f"{get_env_config('config_database')}.t_merge_exclude_consumer_config").filter(F.col("tmec_type") == "ACS")

    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_table = f"{golden_db}.t_master_consumer"
    master_consumer_df = (spark.table(master_consumer_table)
        .join(
            acs_sources_df,
            (F.col("scon_mrkt_code") == F.col("tmec_marketcode")) & (F.col("scon_srcs_code") == F.col("tmec_sourcesystemcode")),
            "left_anti"
        )
    )

    export_keys_df = (
        exploded_df
        .groupBy("MarketCode", "BrandCode", "Ukey")
        .agg(
            F.collect_set(
                F.struct(
                    F.col("SourceSystemCode").alias("SourceSystemCode"),
                    F.col("ConsumerId").alias("ConsumerId")
                )
            ).alias("export_keys")
        )
    )

    master_keys_df = (
        master_consumer_df
        .groupBy(
            F.col("scon_mrkt_code").alias("MarketCode"),
            F.col("scon_brnd_code").alias("BrandCode"),
            F.col("consumermdmkey").alias("Ukey")
        )
        .agg(
            F.collect_set(
                F.struct(
                    F.col("scon_srcs_code").alias("SourceSystemCode"),
                    F.col("scon_consumerid").alias("ConsumerId")
                )
            ).alias("master_keys")
        )
    )

    group_match_df = (
        export_keys_df
        .join(master_keys_df, ["MarketCode", "BrandCode", "Ukey"], "left")
        .withColumn(
            "is_matched",
            F.col("master_keys").isNotNull() &
            (F.size(F.array_except(F.col("export_keys"), F.col("master_keys"))) == 0) &
            (F.size(F.array_except(F.col("master_keys"), F.col("export_keys"))) == 0)
        )
        .select("MarketCode", "BrandCode", "Ukey", "is_matched")
    )

    return (
        exploded_df
        .join(group_match_df, ["MarketCode", "BrandCode", "Ukey"], "left")
        .withColumn(
            "is_matched",
            F.coalesce(F.col("is_matched"), F.lit(False))
        )
    )

In [0]:
def determine_ukey_strategy(matched_df):
    """
    判断 4.1 / 4.2 两种匹配方式：
      4.1: 同一 market + ukey 下存在其他 brand 未在本次 export log 中 → 生成新 UUID
      4.2: 同一 market + ukey 下所有 brand 都在本次 export log 中 → 保持原 Ukey
    """
    golden_db = get_env_config('golden_consumer_master_database')
    master_consumer_df = spark.table(f"{golden_db}.t_master_consumer")

    # 按 market + ukey 聚合 master 中的所有 brand
    master_brands_df = (
        master_consumer_df
        .groupBy("scon_mrkt_code", "consumermdmkey")
        .agg(F.collect_set("scon_brnd_code").alias("master_brands"))
    )

    # 按 market + ukey 聚合本次 export log 中匹配到的 brand
    export_brands_df = (
        matched_df
        .filter(F.col("is_matched"))
        .groupBy("MarketCode", "Ukey")
        .agg(F.collect_set("BrandCode").alias("export_brands"))
    )

    matched_with_brands = (
        matched_df
        .join(
            master_brands_df,
            (F.col("MarketCode") == F.col("scon_mrkt_code")) &
            (F.col("Ukey") == F.col("consumermdmkey")),
            "left"
        )
        .join(export_brands_df, ["MarketCode", "Ukey"], "left")
        .withColumn(
            "has_other_brands",
            F.when(
                F.col("master_brands").isNotNull() & F.col("export_brands").isNotNull(),
                F.size(F.array_except(F.col("master_brands"), F.col("export_brands"))) > 0
            ).otherwise(F.lit(False))
        )
    )

    # 对 case 4.1 的每个 (market, ukey) 生成同一个 UUID（同 ukey 的所有 brand 共享）
    new_ukey_df = (
        matched_with_brands
        .filter(F.col("is_matched") & F.col("has_other_brands"))
        .select("MarketCode", "Ukey")
        .distinct()
        .withColumn("New_UniversalKey", F.expr("uuid()"))
    )

    result_df = (
        matched_with_brands
        .join(new_ukey_df, ["MarketCode", "Ukey"], "left")
        .withColumn(
            "New_UniversalKey",
            F.coalesce(F.col("New_UniversalKey"), F.col("Ukey"))
        )
    )

    return result_df

In [0]:
def write_anonymization_log(result_df, task_id):
    """
    将处理结果写入 t_mdm_anonymization_log，status 直接置为 InProgress。
    """
    anonymization_db = get_env_config('silver_mdm_anonymization_database')
    log_table = f"{anonymization_db}.t_mdm_anonymization_log"

    log_df = result_df.select(
        F.col("ConsumerId"),
        F.col("MarketCode"),
        F.col("BrandCode"),
        F.col("SourceSystemCode"),
        F.col("Ukey").alias("Old_UniversalKey"),
        F.col("New_UniversalKey"),
        F.col("LastActivityTime"),
        F.lit("").alias("comment"),
        F.current_timestamp().alias("create_time"),
        F.lit(task_id).alias("task_id"),
        F.expr("uuid()").alias("cal_uuid"),
        F.lit(ANON_STATUS_IN_PROGRESS).alias("status")
    )

    save_to_target_table(log_df, log_table, f"task_id='{task_id}'")

In [0]:
def generate_anonymization_log(task_id, config_json, recent_days=None):
    """
    步骤 s1.1-s1.5:
      加载 export log → 排除最近已 Complete → 按 market 限流 →
      CID 校验 → 过滤匹配 → 判断 ukey 策略 → 写入 anonymization log (status=InProgress)。

    Args:
        task_id: 任务 ID
        config_json: 限流配置 JSON（格式同原 handle_job_limit 的 config widget）
        recent_days: 排除已 Complete 记录的天数窗口；空/0 读全部
    """
    print("s1.1 load and explode DA export log")

    # 1. 读取并过滤 export_log
    export_log_df = load_export_log()

    if export_log_df.isEmpty():
        print("No export_log records, nothing to do")
        return

    # 2. 展开 ConsumerIdList
    exploded_df = (
        export_log_df
        .select(
            F.col("Ukey"),
            F.col("MarketCode"),
            F.col("BrandCode"),
            F.explode_outer(F.col("ConsumerIdList")).alias("ConsumerId", "SourceSystemCode"),
            F.col("LastActivityTime")
        )
       .where(
            F.col("Ukey").isNotNull() &
            F.col("MarketCode").isNotNull() &
            F.col("BrandCode").isNotNull() &
            F.col("ConsumerId").isNotNull() &
            F.col("SourceSystemCode").isNotNull()
        )
    )

    print("s1.2 load recently completed records")

    # 3. 已 Complete 的 key（recent_days 为空/0 时读全部）
    completed_df = load_recently_completed(recent_days)

    print("s1.3 exclude recently completed keys")

    # 4. 排除已处理 key → Data_02
    data_02 = exploded_df.join(completed_df, ["MarketCode", "BrandCode", "Ukey"], "left_anti")
    data_02 = data_02.checkpoint(eager=True)
    data_02_count = data_02.count()
    print(f'data_02 count: {data_02_count}')

    if data_02_count == 0:
        print("No pending records to process, nothing to do")
        return

    print("s1.4 rate limit per market")

    # 5. 按 market 限流 → Data_03
    data_03 = apply_rate_limit(data_02, config_json)
    if data_03 is None:
        print("No records selected after rate limit, nothing to do")
        return

    data_03 = data_03.checkpoint(eager=True)

    print("s1.5 match and generate new ukey")

    # 6. 与 master_consumer 匹配
    matched_df = match_with_master_consumer(data_03)

    # 7. 只保留通过 CID 校验的记录 → Data_04
    data_04 = matched_df.filter(F.col("is_matched"))
    data_04 = data_04.checkpoint(eager=True)
    data_04_count = data_04.count()
    print(f'data_04 matched count: {data_04_count}')

    if data_04_count == 0:
        print("No matched records, nothing to do")
        return

    # 8. 判断 4.1 / 4.2 并生成 New_UniversalKey
    result_df = determine_ukey_strategy(data_04)

    result_df = result_df.checkpoint(eager=True)
    result_count = result_df.count()
    print(f'total count: {result_count}')

    print("write consumer anonymization log")

    # 9. 写入 t_mdm_anonymization_log
    write_anonymization_log(result_df, task_id)

In [0]:
task_id = dbutils.widgets.get("task_id")
config = dbutils.widgets.get("config")
recent_days = get_ex_param("recent_days", "")
print(f"task_id: {task_id}")
print(f"config: {config}")
print(f"recent_days: {recent_days}")

step_name = "generate_anonymization_log"
step_num = "01"
project = "dataanonymization"
log_table_name = f"{get_env_config('config_database')}.t_task_step_log"

start_time = datetime.now()
status = "SUCCESS"
message = "completed"

try:
    spark.sparkContext.setCheckpointDir(f"{get_env_config('checkpoint_path_consumer_master')}/{task_id}")
    generate_anonymization_log(task_id, config, recent_days)
except Exception as e:
    status = "FAILED"
    message = f"{type(e).__name__}: {str(e)}"
    raise
finally:
    end_time = datetime.now()
    append_step_log(
        log_table_name=log_table_name,
        task_id=task_id,
        step_num=step_num,
        step_name=step_name,
        start_time=start_time,
        end_time=end_time,
        status=status,
        message=message,
        project=project
    )